<a href="https://colab.research.google.com/github/Narendra725/Power_BI_Spark_Labs/blob/main/Power%20BI/Automations/Power%20Bi%20Desktop/Mach3/report_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CLONE GITHUB REPO**

In [ ]:
import os
import sys
from google.colab import userdata

# 1. Configuration
USERNAME = "Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"
ROOT_PATH = f'/content/{REPO_NAME}'

try:
    token = userdata.get('GITHUB_TOKEN')
    AUTH_REPO_URL = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

    # 2. Clean/Sync Repository
    if not os.path.exists(ROOT_PATH):
        !git clone {AUTH_REPO_URL}
    else:
        %cd {ROOT_PATH}
        !git remote set-url origin {AUTH_REPO_URL}
        !git fetch origin
        !git reset --hard origin/main

    # 3. Path Initialization
    if ROOT_PATH not in sys.path:
        sys.path.append(ROOT_PATH)

    MACH3_ROOT = os.path.join(ROOT_PATH, 'Power BI/Automations/Power Bi Desktop/Mach3')
    %cd "{MACH3_ROOT}"

    print(f"Environment Reinitialized.\nRoot: {ROOT_PATH}\nWorking Dir: {os.getcwd()}")
except Exception as e:
    print(f"Initialization Error: {e}")

/content/Power_BI_Spark_Labs
HEAD is now at 009ad2b Created using Colab
/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3
Environment Reinitialized.
Root: /content/Power_BI_Spark_Labs
Working Dir: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3


# **FUNCTIONS DECLARATION**

In [108]:
import zipfile
import shutil
import os
import json
import pandas as pd
from mach3_core import FabricReport
from fabric_models import Report, Page, VisualContainer, Bookmark

def check_models_gen():
  model_path = '/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/fabric_models.py'
  with open(model_path, 'r') as f:
      first_lines = [next(f) for _ in range(1)]
  print(''.join(first_lines))
  return

def delete(folder_path):
  if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
  return f"{folder_path} deleted"

def zip(folder_path, zip_path):
    shutil.make_archive(zip_path, 'zip', folder_path)
    return zip_path + '.zip' + 'created'

def unzip(zip_path = '/content/definition.zip', extract_path = '/content/definition'):
  if os.path.exists(zip_path):
      with zipfile.ZipFile(zip_path, 'r') as zip_ref:
          zip_ref.extractall(extract_path)
      print(f"Extracted {zip_path} to {extract_path}/")
  else:
      print(f"Zip file not found at {zip_path}. Checking if definition folder exists directly...")
      if os.path.exists(zip_path):
          print("Found unzipped folder in repository.")
      else:
          print("Could not find zipfile/folder at the specified zip_path.")

class FabricBPARules:
    def __init__(self, report_obj):
        self.report = report_obj
        self.results = []
        self.errors = []

    @staticmethod
    def load_from_folder(src_path):
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"Folder not found: {src_path}")
        pages_list = []
        bookmarks_list = []
        with open(os.path.join(src_path, 'report.json'), 'r') as f:
            master_report = Report(**json.load(f))
        pages_dir = os.path.join(src_path, 'pages')
        if os.path.exists(pages_dir):
            for p_folder in os.listdir(pages_dir):
                folder_path = os.path.join(pages_dir, p_folder)
                if not os.path.isdir(folder_path): continue
                page_json_path = os.path.join(folder_path, 'page.json')
                if os.path.exists(page_json_path):
                    with open(page_json_path, 'r') as f:
                        page_obj = Page(**json.load(f))
                    v_list = []
                    v_dir = os.path.join(folder_path, 'visuals')
                    if os.path.exists(v_dir):
                        for v_f in os.listdir(v_dir):
                            v_path = os.path.join(v_dir, v_f, 'visual.json')
                            if os.path.exists(v_path):
                                with open(v_path, 'r') as f:
                                    v_list.append(VisualContainer(**json.load(f)))
                    pages_list.append((page_obj, v_list))
        bookmarks_dir = os.path.join(src_path, 'bookmarks')
        if os.path.exists(bookmarks_dir):
            for b_file in os.listdir(bookmarks_dir):
                if b_file.endswith('.bookmark.json'):
                    with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
                        bookmarks_list.append(Bookmark(**json.load(f)))
        return FabricReport(master_report, pages_list, bookmarks_list)

    def _get_visual_metadata(self, visual):
        v_data = visual.root.visual
        if v_data is None:
            return "unknown", ""
        if hasattr(v_data, 'model_dump'):
            v_data = v_data.model_dump(by_alias=True, exclude_none=True)
        v_type = v_data.get('visualType', v_data.get('visual_type', 'unknown'))
        v_display = ""
        try:
            title_obj = v_data.get('visualContainerObjects', {}).get('title', [])[0]
            v_display = title_obj.get('properties', {}).get('text', {}).get('expr', {}).get('Literal', {}).get('Value', '')
            v_display = v_display.strip("'")
        except (IndexError, AttributeError, KeyError):
            v_display = ""
        return v_type, v_display

    def fix_page_names(self, src_path):
        """Renames internal page IDs and synchronizes all navigation references."""
        pages_json_path = os.path.join(src_path, 'pages', 'pages.json')
        if not os.path.exists(pages_json_path):
            ordered_ids = [p.model.name for p in self.report.pages]
        else:
            with open(pages_json_path, 'r') as f:
                ordered_ids = json.load(f).get('pageOrder', []) # Changed to pageOrder

        self.report.pages.sort(key=lambda p: ordered_ids.index(p.model.name) if p.model.name in ordered_ids else 999)

        page_id_rename_map = {}
        for i, page_wrapper in enumerate(self.report.pages, 1):
            old_id = page_wrapper.model.name
            new_id = f"Page{i}"
            if old_id != new_id:
                page_id_rename_map[old_id] = new_id
                page_wrapper.model.name = new_id

        if page_id_rename_map:
            # 1. Update Bookmark references to Page IDs
            for bookmark in self.report.bookmarks:
                bm_json = json.dumps(bookmark.model_dump(by_alias=True, mode='json'))
                for old, new in page_id_rename_map.items():
                    bm_json = bm_json.replace(f'"{old}"', f'"{new}"')
                updated_bm = Bookmark(**json.loads(bm_json))
                bookmark.__dict__.update(updated_bm.__dict__)

            # 2. Update Visual Navigation references (Buttons/Actions) to Page IDs
            for page_wrapper in self.report.pages:
                for visual in page_wrapper.visuals:
                    v_json = json.dumps(visual.model_dump(by_alias=True, mode='json'))
                    for old, new in page_id_rename_map.items():
                        v_json = v_json.replace(f'"{old}"', f'"{new}"')
                    updated_v = VisualContainer(**json.loads(v_json))
                    visual.__dict__.update(updated_v.__dict__)

        print(f"Renamed {len(page_id_rename_map)} internal page IDs and synchronized navigation.")
        return page_id_rename_map

    def fix_visual_names(self):
        rename_map = {}
        for page_wrapper in self.report.pages:
            page_display = page_wrapper.model.displayName.replace(' ', '')
            visual_counter = 1
            for visual in page_wrapper.visuals:
                old_name = visual.root.name
                v_type, v_display = self._get_visual_metadata(visual)
                short_display = "".join(e for e in v_display if e.isalnum())[:15]
                if not short_display:
                    short_display = "visual"
                new_name = f"{page_display}_{v_type}_{short_display}_{visual_counter}"
                if old_name != new_name:
                    visual.root.name = new_name
                    rename_map[old_name] = new_name
                visual_counter += 1
        for page_wrapper in self.report.pages:
            for visual in page_wrapper.visuals:
                if hasattr(visual.root, 'parent_group_name') and visual.root.parent_group_name in rename_map:
                    visual.root.parent_group_name = rename_map[visual.root.parent_group_name]
        if rename_map:
            for bookmark in self.report.bookmarks:
                bookmark_str = json.dumps(bookmark.model_dump(by_alias=True, mode='json'))
                for old, new in rename_map.items():
                    bookmark_str = bookmark_str.replace(f'"{old}"', f'"{new}"')
                updated_bm = Bookmark(**json.loads(bookmark_str))
                bookmark.__dict__.update(updated_bm.__dict__)
        print(f"Renamed {len(rename_map)} visuals.")
        return rename_map

    def save_report(self, output_path):
        if os.path.exists(output_path):
            shutil.rmtree(output_path)
        os.makedirs(output_path)
        master = getattr(self.report, 'metadata', getattr(self.report, 'report', None))
        if master:
            with open(os.path.join(output_path, 'report.json'), 'w') as f:
                json.dump(master.model_dump(by_alias=True, mode='json'), f, indent=2)
        pages_dir = os.path.join(output_path, 'pages')
        os.makedirs(pages_dir)
        # Corrected pages.json structure to match the Power BI schema
        pages_index = {
            "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json",
            "pageOrder": [p.model.name for p in self.report.pages],
            "activePageName": self.report.pages[0].model.name if self.report.pages else None # Assuming the first page is active, or None if no pages
        }
        with open(os.path.join(pages_dir, 'pages.json'), 'w') as f:
            json.dump(pages_index, f, indent=2)
        for p_wrapper in self.report.pages:
            p_id = p_wrapper.model.name
            p_path = os.path.join(pages_dir, p_id)
            os.makedirs(p_path)
            with open(os.path.join(p_path, 'page.json'), 'w') as f:
                json.dump(p_wrapper.model.model_dump(by_alias=True, mode='json'), f, indent=2)
            if p_wrapper.visuals:
                v_dir = os.path.join(p_path, 'visuals')
                os.makedirs(v_dir)
                for visual in p_wrapper.visuals:
                    v_id = visual.root.name
                    v_path = os.path.join(v_dir, v_id)
                    os.makedirs(v_path)
                    with open(os.path.join(v_path, 'visual.json'), 'w') as f:
                        json.dump(visual.model_dump(by_alias=True, mode='json'), f, indent=2)
        if self.report.bookmarks:
            bookmarks_dir = os.path.join(output_path, 'bookmarks')
            os.makedirs(bookmarks_dir)
            bookmarks_index = {"bookmarks": [bm.name for bm in self.report.bookmarks]}
            with open(os.path.join(bookmarks_dir, 'bookmarks.json'), 'w') as f:
                json.dump(bookmarks_index, f, indent=2)
            for bm in self.report.bookmarks:
                with open(os.path.join(bookmarks_dir, f'{bm.name}.bookmark.json'), 'w') as f:
                    json.dump(bm.model_dump(by_alias=True, mode='json'), f, indent=2)

In [ ]:
def validate_report_schema(fabric_report_obj):
    print("--- Starting Schema Validation ---")
    try:
        if hasattr(fabric_report_obj, 'metadata') and fabric_report_obj.metadata:
            dump = fabric_report_obj.metadata.model_dump(by_alias=True)
            fabric_report_obj.metadata.model_validate(dump)
            print(f"✓ Metadata Schema: Valid")
        for page_wrapper in fabric_report_obj.pages:
            page_wrapper.model.model_validate(page_wrapper.model.model_dump(by_alias=True))
            for v in page_wrapper.visuals:
                v.model_validate(v.model_dump(by_alias=True))
        print(f"✓ {len(fabric_report_obj.pages)} Pages and associated Visuals: Valid")
        for bookmark in fabric_report_obj.bookmarks:
            bookmark.model_validate(bookmark.model_dump(by_alias=True))
        print(f"✓ {len(fabric_report_obj.bookmarks)} Bookmarks: Valid")
        print("\nSUCCESS: All components adhere to the fabric_models schema.")
        return True
    except Exception as e:
        print(f"\nSCHEMA VALIDATION FAILED:")
        print(str(e))
        return False

# **GENERATE MODELS**

In [ ]:
%run power_bi_objects_creation.ipynb

# **IMPORT OBJECTS**

In [ ]:
from fabric_models import Report, Page, VisualContainer, Bookmark
from mach3_core import  FabricReport,FabricBPARules
import os
import json


In [ ]:
check_models_gen()

# Generated on: 2026-04-30 14:34:56 IST



In [ ]:
import inspect
from fabric_models import Page
# Inspect the Page model to see available attributes
print(f"Attributes in Page model: {list(Page.__annotations__.keys())}")

In [ ]:
# Execute the inspection to see Page model fields
from fabric_models import Page
import json

try:
    # Try to see fields and their aliases
    print("Page Model Fields:", Page.model_fields.keys())
except:
    print("Annotations:", Page.__annotations__.keys())


Page Model Fields: dict_keys(['field_schema', 'name', 'displayName', 'displayOption', 'height', 'width', 'filterConfig', 'pageBinding', 'objects', 'type', 'visibility', 'visualInteractions', 'autoPageGenerationConfig', 'annotations', 'howCreated'])


In [ ]:
from fabric_models import VisualContainer
try:
    print("VisualContainer Model Fields:", VisualContainer.model_fields.keys())
except:
    print("Annotations:", VisualContainer.__annotations__.keys())

VisualContainer Model Fields: dict_keys(['root'])


In [ ]:
delete('/content/definition')

'/content/definition deleted'

## **Zip and UnZip Folders**

In [ ]:
zip( folder_path= '/content/definition', zip_path= '/content')

In [ ]:
unzip(zip_path = '/content/definition.zip',extract_path='/content')

Extracted /content/definition.zip to /content/


# **REPORT GEN AND TESTING**

## **Parse/ Load the Report from definition Folder**

In [ ]:
# src_path = os.path.join(MACH3_ROOT, 'definition')
# pages_list = []
# bookmarks_list = []

# if os.path.exists(src_path):
#     print(f"Loading definition from: {src_path}")
#     try:
#         with open(os.path.join(src_path, 'report.json'), 'r') as f:
#             master_report = Report(**json.load(f))

#         pages_dir = os.path.join(src_path, 'pages')
#         if os.path.exists(pages_dir):
#             for p_folder in os.listdir(pages_dir):
#                 folder_path = os.path.join(pages_dir, p_folder)
#                 if not os.path.isdir(folder_path): continue
#                 page_json_path = os.path.join(folder_path, 'page.json')
#                 if os.path.exists(page_json_path):
#                     with open(page_json_path, 'r') as f:
#                         page_obj = Page(**json.load(f))
#                     v_list = []
#                     v_dir = os.path.join(folder_path, 'visuals')
#                     if os.path.exists(v_dir):
#                         for v_f in os.listdir(v_dir):
#                             v_path = os.path.join(v_dir, v_f, 'visual.json')
#                             if os.path.exists(v_path):
#                                 with open(v_path, 'r') as f:
#                                     v_list.append(VisualContainer(**json.load(f)))
#                     pages_list.append((page_obj, v_list))

#         bookmarks_dir = os.path.join(src_path, 'bookmarks')
#         if os.path.exists(bookmarks_dir):
#             for b_file in os.listdir(bookmarks_dir):
#                 if b_file.endswith('.bookmark.json'):
#                     with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
#                         bookmarks_list.append(Bookmark(**json.load(f)))

#         report = FabricReport(master_report, pages_list, bookmarks_list)
#         print("\nSUCCESS: Report objects created and validated.")
#         report.get_summary()
#     except Exception as e:
#         print(f"Validation error details:\n{e}")
# else:
#     print(f"Definition folder not found at {src_path}.")

Loading definition from: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/definition

SUCCESS: Report objects created and validated.
--- Fabric Report Master Summary ---
Pages: 11 | Bookmarks: 26
- Transaction Trend (29 visuals)
- Private Label (28 visuals)
- Online (32 visuals)
- Insurance (34 visuals)
- Region (25 visuals)
- Units (36 visuals)
- Promo (37 visuals)
- S&OP Achievement (41 visuals)
- Sales (44 visuals)
- Enablers (49 visuals)
- Time-Based Analysis (24 visuals)


In [ ]:
report.get_summary()

--- Fabric Report Master Summary ---
Pages: 11 | Bookmarks: 26
- Promo (37 visuals)
- Enablers (49 visuals)
- Transaction Trend (29 visuals)
- Sales (44 visuals)
- Online (32 visuals)
- Insurance (34 visuals)
- Private Label (28 visuals)
- Time-Based Analysis (24 visuals)
- Units (36 visuals)
- Region (25 visuals)
- S&OP Achievement (41 visuals)


In [ ]:
# Execute the updated validation function
validate_report_schema(report)

--- Starting Schema Validation ---
✓ Metadata Schema: Valid
✓ 11 Pages and associated Visuals: Valid
✓ 26 Bookmarks: Valid

SUCCESS: All components adhere to the fabric_models schema.


True

# **BPA RULES ENGINE DEFINITION**

### **How to use FabricBPARules**

To initialize and run the rules engine, follow these steps:

1. **Initialize**: Pass your `report` object to `FabricBPARules`.
2. **Validate**: Call `.validate_schema()` to ensure the report structure is correct.
3. **Analyze**: Call `.run_all_checks()` to find naming convention issues.
4. **Fix & Save**: Use `.fix_visual_names()` to rename and `.save_report()` to export the results.

In [107]:
pages_json_path = os.path.join('/content/definition', 'pages', 'pages.json')
if os.path.exists(pages_json_path):
    with open(pages_json_path, 'r') as f:
        original_pages_json = json.load(f)
    print("Original pages.json content:")
    display(original_pages_json)
else:
    print(f"File not found: {pages_json_path}")

Original pages.json content:


{'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json',
 'pageOrder': ['SalesProject_Financials_Page_2',
  'SalesProject_Financials_Page_3'],
 'activePageName': 'SalesProject_Financials_Page_2'}

In [109]:
# 1. Reload the report fresh
report_folder = '/content/definition'
report_obj = FabricBPARules.load_from_folder(report_folder)
bpa_engine = FabricBPARules(report_obj)

# 2. Rename internal Page Names (IDs) based on pages.json order
# This changes the GUIDs to Page1, Page2, etc.
page_id_renames = bpa_engine.fix_page_names(report_folder)

print("\nInternal Page ID Renaming Audit:")
for old_id, new_id in page_id_renames.items():
    print(f"  - ID: {old_id} -> {new_id}")

# 3. Run visual renaming logic
# Note: This still uses the display names for the visual prefix as per the rule
visual_renames = bpa_engine.fix_visual_names()

# 4. Save the definition - this will create folders named Page1, Page2, etc.
output_report_path = '/content/fixed_definition_v6'
bpa_engine.save_report(output_report_path)

print(f"\nSUCCESS: Report with renamed internal IDs saved to: {output_report_path}")

Renamed 2 internal page IDs and synchronized navigation.

Internal Page ID Renaming Audit:
  - ID: SalesProject_Financials_Page_2 -> Page1
  - ID: SalesProject_Financials_Page_3 -> Page2
Renamed 40 visuals.

SUCCESS: Report with renamed internal IDs saved to: /content/fixed_definition_v6


In [110]:
fixed_pages_json_path = os.path.join('/content/fixed_definition_v6', 'pages', 'pages.json')
if os.path.exists(fixed_pages_json_path):
    with open(fixed_pages_json_path, 'r') as f:
        fixed_pages_json = json.load(f)
    print("Fixed pages.json content in /content/fixed_definition_v6:")
    display(fixed_pages_json)
else:
    print(f"File not found: {fixed_pages_json_path}")

Fixed pages.json content in /content/fixed_definition_v6:


{'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json',
 'pageOrder': ['Page1', 'Page2'],
 'activePageName': 'Page1'}

In [ ]:
# 1. Reload the report fresh
report_obj = FabricBPARules.load_from_folder(report_folder)
bpa_engine = FabricBPARules(report_obj)

# 2. Rename internal Page Names (IDs) based on pages.json order
# This changes the GUIDs to Page1, Page2, etc.
page_id_renames = bpa_engine.fix_page_names(report_folder)

print("\nInternal Page ID Renaming Audit:")
for old_id, new_id in page_id_renames.items():
    print(f"  - ID: {old_id} -> {new_id}")

# 3. Run visual renaming logic
# Note: This still uses the display names for the visual prefix as per the rule
visual_renames = bpa_engine.fix_visual_names()

# 4. Save the definition - this will create folders named Page1, Page2, etc.
output_report_path = '/content/fixed_definition_v6'
bpa_engine.save_report(output_report_path)

print(f"\nSUCCESS: Report with renamed internal IDs saved to: {output_report_path}")

Renamed 11 internal page IDs (GUIDs -> PageX).

Internal Page ID Renaming Audit:
  - ID: 2390b9dabcd5ce11f229 -> Page1
  - ID: ba72687cd95134c9cf50 -> Page2
  - ID: ecd721571ed7003ea3d2 -> Page3
  - ID: 4acdeb112c1f82d46871 -> Page4
  - ID: 47a1e4e372f625f273fd -> Page5
  - ID: e01e22ea8fc1486624e4 -> Page6
  - ID: 0b8572c658728c3b9105 -> Page7
  - ID: 839e46f1e69e5b159269 -> Page8
  - ID: 8f5df3e31a9b7a6cbe75 -> Page9
  - ID: 9f92c5b236d3b9200479 -> Page10
  - ID: 61bb103bb90dd00a7407 -> Page11
Renamed 379 visuals.

SUCCESS: Report with renamed internal IDs saved to: /content/fixed_definition_v6


In [ ]:
import json

def audit_bookmark_links(report_obj, num_to_check=3):
    print(f"--- Auditing Bookmark-to-Page Links ---")
    checked = 0
    for bm in report_obj.bookmarks:
        # Convert model back to dict to see raw JSON structure
        bm_dict = bm.model_dump(by_alias=True, mode='json')
        bm_str = json.dumps(bm_dict)

        # Check if the bookmark contains a reference to the new Page naming convention
        if "Page" in bm_str and checked < num_to_check:
            print(f"\nBookmark Name: {bm.name}")
            # Print a snippet of the JSON where the page reference usually lives
            # We'll display the whole thing if it's small, or just look for 'Page'
            display(bm_dict)
            checked += 1

    if checked == 0:
        print("No Page references found in bookmarks. This might mean the bookmarks don't have page-specific state.")

audit_bookmark_links(bpa_engine.report)

--- Auditing Bookmark-to-Page Links ---

Bookmark Name: c57f7fe4dac90490ca42


{'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/bookmark/1.4.0/schema.json',
 'displayName': 'Hide Filters S&OP',
 'name': 'c57f7fe4dac90490ca42',
 'options': {'targetVisualNames': [], 'suppressData': True},
 'explorationState': {'version': '1.11',
  'activeSection': 'Page8',
  'filters': {'byName': {'98d872e30db7b1e2c065': {'name': '98d872e30db7b1e2c065',
     'type': 'RelativeDate',
     'filter': {'Version': 2,
      'From': [{'Name': 'd', 'Entity': 'Date', 'Type': 0}],
      'Where': [{'Condition': {'Comparison': {'ComparisonKind': 0,
          'Left': {'Column': {'Expression': {'SourceRef': {'Source': 'd'}},
            'Property': 'Date'}},
          'Right': {'DateSpan': {'Expression': {'Now': {}},
            'TimeUnit': 3}}}}}]},
     'expression': {'Column': {'Expression': {'SourceRef': {'Entity': 'Date'}},
       'Property': 'Date'}},
     'howCreated': 1},
    'da42ae80de672da6a0ce': {'name': 'da42ae80de672da6a0ce',
     'type': 'Cate


Bookmark Name: 6c1a242a31167863b1e4


{'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/bookmark/1.4.0/schema.json',
 'displayName': 'Hide Filters Enablers',
 'name': '6c1a242a31167863b1e4',
 'options': {'applyOnlyToTargetVisuals': True,
  'targetVisualNames': ['Enablers_unknown_visual_20',
   'Enablers_actionButton_clearslicers_25',
   'Enablers_slicer_visual_33',
   'Enablers_slicer_visual_40',
   'Enablers_slicer_visual_35',
   'Enablers_shape_visual_24',
   'Enablers_slicer_visual_45',
   'Enablers_actionButton_visual_9',
   'Enablers_slicer_visual_2',
   'Enablers_slicer_visual_14',
   'Enablers_slicer_visual_29',
   'Enablers_image_visual_17',
   'Enablers_slicer_visual_43'],
  'suppressData': True},
 'explorationState': {'version': '1.11',
  'activeSection': 'Page10',
  'filters': {'byName': {'98d872e30db7b1e2c065': {'name': '98d872e30db7b1e2c065',
     'type': 'Advanced',
     'filter': {'Version': 2,
      'From': [{'Name': 'd', 'Entity': 'Date', 'Type': 0}],
      'Where': [{


Bookmark Name: ec0976c85b1e4dead050


{'$schema': 'https://developer.microsoft.com/json-schemas/fabric/item/report/definition/bookmark/1.4.0/schema.json',
 'displayName': 'Show Filters Transaction Trend',
 'name': 'ec0976c85b1e4dead050',
 'options': {'targetVisualNames': ['TransactionTrend_actionButton_clearslicers_16'],
  'suppressData': True},
 'explorationState': {'version': '1.11',
  'activeSection': 'Page1',
  'filters': {'byName': {'da42ae80de672da6a0ce': {'name': 'da42ae80de672da6a0ce',
     'type': 'Categorical',
     'expression': {'Column': {'Expression': {'SourceRef': {'Entity': 'Date'}},
       'Property': 'Future Dates Hidden'}},
     'howCreated': 1}},
   'byExpr': [{'name': '98d872e30db7b1e2c065',
     'type': 'RelativeDate',
     'filter': {'Version': 2,
      'From': [{'Name': 'd', 'Entity': 'Date', 'Type': 0}],
      'Where': [{'Condition': {'Comparison': {'ComparisonKind': 0,
          'Left': {'Column': {'Expression': {'SourceRef': {'Source': 'd'}},
            'Property': 'Date'}},
          'Right': {

In [ ]:
def audit_all_visual_actions(report_obj, num_to_check=10):
    print('--- Auditing All Visual Actions (Buttons/Images/Shapes) ---')
    found_actions = 0

    for page_wrapper in report_obj.pages:
        for visual in page_wrapper.visuals:
            v_dict = visual.model_dump(by_alias=True, mode='json')
            # Deep scan for 'action' objects which hold navigation metadata
            actions = v_dict.get('root', {}).get('visual', {}).get('visualContainerObjects', {}).get('action', [])

            if actions and found_actions < num_to_check:
                print(f'\n[Page: {page_wrapper.model.name}] Visual: {visual.root.name}')

                for action in actions:
                    properties = action.get('properties', {})
                    # Extract type (e.g., Bookmark, PageNavigation, Back, WebURL)
                    action_type = properties.get('type', {}).get('expr', {}).get('Literal', {}).get('Value', 'Unknown')
                    print(f'  - Action Type: {action_type}')

                    # Check for Bookmark target mapping
                    if 'bookmark' in properties:
                        bm_target = properties.get('bookmark', {}).get('expr', {}).get('Literal', {}).get('Value', 'N/A')
                        print(f'  - Bookmark Target: {bm_target}')

                    # Check for Section/Page target mapping
                    if 'section' in properties:
                        page_target = properties.get('section', {}).get('expr', {}).get('Literal', {}).get('Value', 'N/A')
                        print(f'  - Page Target: {page_target}')

                display(v_dict)
                found_actions += 1

    if found_actions == 0:
        print('No interactive actions found in any visual containers.')

audit_all_visual_actions(bpa_engine.report)

--- Auditing All Visual Actions (Buttons/Images/Shapes) ---
No interactive actions found in any visual containers.


# **PUSH CHANGES TO GIT**

In [ ]:
# push to git
from mach3_helpers import push_to_github
push_to_github(ROOT_PATH,commit_message='Save the models')

Push sequence complete.


Project Context: Power BI PBIR Automation
We are developing a programmatic flow to manage Power BI reports in the Fabric Enhanced Report Format (PBIR) using Python and Pydantic.

Current Progress:
Environment Setup: We've successfully cloned the Power_BI_Spark_Labs repository and initialized a core engine (FabricBPARules) that uses dynamically generated Pydantic models to represent report components (Report, Page, Visuals).
BPA Implementation: We successfully implemented Best Practice Analyzer (BPA) rules that rename report pages (e.g., Page1, Page2) and visuals sequentially to ensure consistency.
Schema Fix: We identified and fixed a schema violation in pages.json within the save_report method of the FabricBPARules class. The pages.json file now correctly outputs pageOrder and activePageName properties, aligning with the pagesMetadata schema.
Next Steps:
Verification: Load the processed report from /content/fixed_definition_v6 into Power BI Desktop to confirm that all pages and visuals are displayed correctly and that the 'Frown' error is resolved.
Git Sync: Once verified in Power BI Desktop, we will move the finalized report to the templates folder and push the changes to GitHub.
You can resume from here tomorrow!